# Séance 2 · Python pour la data · ⭐

**Niveau : ⭐ Débutant**

On va analyser un vrai jeu de données : les 800 Pokémon avec leurs statistiques. Deux outils : **pandas** (tableaux de données)
et **matplotlib** (graphiques). Ce notebook tourne dans **Google Colab** : rien à installer. Pour exécuter une cellule : `Maj + Entrée`.

**Livrable de la séance** : un notebook avec 3 questions posées sur un dataset et 3 graphiques qui y répondent.


## Préparation

On importe pandas et matplotlib (déjà dans Colab) et on charge le dataset Pokémon depuis Internet.
Si le réseau ne répond pas, une mini-version de secours est créée pour que la suite fonctionne quand même.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    df = pd.read_csv(url)
    print("Dataset Pokémon chargé :", df.shape[0], "lignes")
except Exception as e:
    print("Pas de réseau ? On utilise une mini-version de secours.", e)
    df = pd.DataFrame({
        "#": [1, 4, 7, 25, 94, 130, 143, 150], "Name": ["Bulbasaur", "Charmander", "Squirtle", "Pikachu", "Gengar", "Gyarados", "Snorlax", "Mewtwo"],
        "Type 1": ["Grass", "Fire", "Water", "Electric", "Ghost", "Water", "Normal", "Psychic"], "Type 2": ["Poison", None, None, None, "Poison", "Flying", None, None],
        "Total": [318, 309, 314, 320, 500, 540, 540, 680], "HP": [45, 39, 44, 35, 60, 95, 160, 106], "Attack": [49, 52, 48, 55, 65, 125, 110, 110],
        "Defense": [49, 43, 65, 40, 60, 79, 65, 90], "Sp. Atk": [65, 60, 50, 50, 130, 60, 65, 154], "Sp. Def": [65, 50, 64, 50, 75, 100, 110, 90],
        "Speed": [45, 65, 43, 90, 110, 81, 30, 130], "Generation": [1] * 8, "Legendary": [False] * 7 + [True],
    })
df.head()

## 1. Rappels rapides : fonctions, listes, dictionnaires, boucles (10 min)

Tout ce dont on a besoin en data tient en 4 briques. Petit tour de chauffe avant pandas.

In [ ]:
# Liste : plusieurs valeurs dans l'ordre
vitesses = [45, 65, 43, 90, 110]

# Boucle : on passe sur chaque valeur
total = 0
for v in vitesses:
    total += v
print("moyenne à la main :", total / len(vitesses))

# Fonction : une recette réutilisable
def moyenne(liste):
    return sum(liste) / len(liste)

print("moyenne avec la fonction :", moyenne(vitesses))

In [ ]:
# Dictionnaire : clé -> valeur (une fiche Pokémon, c'est un dictionnaire)
pikachu = {"nom": "Pikachu", "type": "Electric", "vitesse": 90, "attaque": 55}
print(pikachu["nom"], "est de type", pikachu["type"])

# Liste de dictionnaires : plusieurs fiches (c'est déjà presque un tableau !)
equipe = [pikachu, {"nom": "Snorlax", "type": "Normal", "vitesse": 30, "attaque": 110}]
for p in equipe:
    print(p["nom"], "->", p["vitesse"])

**Exercice** : écris `plus_rapide(equipe)` qui renvoie le **nom** du Pokémon le plus rapide de la liste `equipe`.
Indice : garde en mémoire le meilleur trouvé jusqu'ici dans la boucle.

In [ ]:
# À toi
def plus_rapide(equipe):
    meilleur = equipe[0]
    return meilleur["nom"]

print(plus_rapide(equipe))   # attendu : Pikachu

<details><summary>Solution</summary>

```python
def plus_rapide(equipe):
    meilleur = equipe[0]
    for p in equipe:
        if p["vitesse"] > meilleur["vitesse"]:
            meilleur = p
    return meilleur["nom"]

print(plus_rapide(equipe))
```
</details>

## 2. Découvrir un tableau avec pandas

`df` est un **DataFrame** : un tableau avec des lignes (les Pokémon) et des colonnes (les stats), comme une feuille Excel qu'on manipule en Python.
Une colonne seule (`df["Speed"]`) s'appelle une **Series**.

In [ ]:
print("Lignes et colonnes :", df.shape)
print("Colonnes :", df.columns.tolist())
df.head(3)

In [ ]:
df.info()          # type de chaque colonne, valeurs manquantes
df.describe()      # min, max, moyenne... de chaque colonne de nombres

**Exercice** : affiche les 5 **dernières** lignes (`tail`), puis uniquement les colonnes `Name`, `Type 1` et `Speed` des 10 premières lignes.
Indice : `df[["Name", "Type 1", "Speed"]]` sélectionne plusieurs colonnes.

In [ ]:
# À toi
df.head(2)

<details><summary>Solution</summary>

```python
print(df.tail())
df[["Name", "Type 1", "Speed"]].head(10)
```
</details>

## 3. Filtrer

Filtrer, c'est poser une condition et ne garder que les lignes qui la respectent : `df[condition]`.
Pour combiner : `&` (et), `|` (ou), avec des parenthèses autour de chaque condition.

In [ ]:
feu = df[df["Type 1"] == "Fire"]
print("Pokémon de type Feu :", len(feu))

rapides_et_forts = df[(df["Speed"] > 100) & (df["Attack"] > 100)]
print("Rapides ET forts :", len(rapides_et_forts))
rapides_et_forts[["Name", "Type 1", "Attack", "Speed"]].head()

**Exercice** : compte les Pokémon **légendaires** (`Legendary == True`), puis affiche les Pokémon de type Eau (`Water`) de la génération 1 avec plus de 90 en défense.

In [ ]:
# À toi
legendaires = df[df["Legendary"] == True]
print(len(legendaires))

<details><summary>Solution</summary>

```python
legendaires = df[df["Legendary"] == True]
print("Légendaires :", len(legendaires))

eau_g1 = df[(df["Type 1"] == "Water") & (df["Generation"] == 1) & (df["Defense"] > 90)]
eau_g1[["Name", "Defense"]]
```
</details>

## 4. Trier et compter

- `sort_values("colonne")` trie (ajoute `ascending=False` pour du plus grand au plus petit).
- `value_counts()` compte combien de fois chaque valeur apparaît : parfait pour les catégories.

In [ ]:
plus_rapides = df.sort_values("Speed", ascending=False).head(5)
plus_rapides[["Name", "Type 1", "Speed"]]

In [ ]:
df["Type 1"].value_counts().head(8)     # les 8 types les plus fréquents

**Exercice** : affiche les 5 Pokémon avec la meilleure attaque (`Attack`), puis compte les Pokémon par génération.

In [ ]:
# À toi
df.sort_values("Attack").head(5)[["Name", "Attack"]]

<details><summary>Solution</summary>

```python
print(df.sort_values("Attack", ascending=False).head(5)[["Name", "Attack"]])
print(df["Generation"].value_counts().sort_index())
```
</details>

## 5. Regrouper avec `groupby`

`groupby` fait des tas : un tas par catégorie (par type, par génération...), puis calcule quelque chose sur chaque tas : moyenne, total, compte, max.
Analogie : trier ses cartes par couleur, puis calculer la valeur moyenne de chaque paquet.

In [ ]:
par_type = df.groupby("Type 1")["Total"].mean().sort_values(ascending=False)
par_type.round(1)

In [ ]:
# Plusieurs calculs d'un coup
df.groupby("Generation").agg(nb_pokemon=("Name", "count"), vitesse_moyenne=("Speed", "mean"), attaque_max=("Attack", "max")).round(1)

**Exercice** : quelle est la vie moyenne (`HP`) par type ? Et les légendaires ont-ils vraiment un `Total` plus élevé que les autres ? (groupby sur `Legendary`)

In [ ]:
# À toi
df.groupby("Type 1")["HP"].count().head()

<details><summary>Solution</summary>

```python
print(df.groupby("Type 1")["HP"].mean().sort_values(ascending=False).round(1))
print(df.groupby("Legendary")["Total"].mean().round(1))
```
</details>

## 6. Trois types de graphiques

Un graphique = une question + le bon type de dessin :

| Question | Graphique | Code |
|---|---|---|
| Comparer des catégories | **barres** | `.plot(kind="bar")` |
| Y a-t-il un lien entre 2 nombres ? | **nuage de points** | `plt.scatter(x, y)` |
| Comment une valeur est-elle répartie ? | **histogramme** | `.plot(kind="hist")` |

In [ ]:
par_type.plot(kind="bar", figsize=(10, 4), title="Puissance moyenne par type")
plt.ylabel("Total des stats")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["Attack"], df["Defense"], alpha=0.5)
plt.xlabel("Attaque")
plt.ylabel("Défense")
plt.title("Attaque vs Défense : y a-t-il un lien ?")
plt.show()

In [ ]:
df["Speed"].plot(kind="hist", bins=20, figsize=(7, 4), title="Répartition de la vitesse")
plt.xlabel("Vitesse")
plt.show()

**Exercice** : trace un histogramme de la colonne `HP`. Que remarques-tu ? Puis un graphique en barres du nombre de Pokémon par génération.
Indice : `value_counts().sort_index().plot(kind="bar")`.

In [ ]:
# À toi
df["HP"].describe()

<details><summary>Solution</summary>

```python
df["HP"].plot(kind="hist", bins=20, title="Répartition des HP")
plt.show()
# La plupart des Pokémon ont entre 40 et 100 HP ; quelques monstres dépassent 200 (Blissey !)

df["Generation"].value_counts().sort_index().plot(kind="bar", title="Pokémon par génération")
plt.show()
```
</details>

## 7. Projet : 3 questions, 3 graphiques (80 min)

C'est ton premier projet de portfolio. La règle du jeu :

1. Choisis **3 questions** qui t'intéressent vraiment sur le dataset (Pokémon par défaut, ou un autre : voir section 8).
2. Pour chaque question : une cellule de calcul (filtre / tri / groupby) et **un graphique** qui y répond.
3. Sous chaque graphique, écris la réponse en une phrase dans le commentaire `# Réponse :`.

Exemples de questions : les légendaires sont-ils vraiment plus forts ? Quelle génération a les Pokémon les plus rapides ?
Quel type a le plus de vie ? Les Pokémon rapides sont-ils fragiles ? Quel est le type le plus rare ?

Écris d'abord tes 3 questions ici :

- Question 1 : ...
- Question 2 : ...
- Question 3 : ...

In [ ]:
# Question 1 : (écris-la ici)
# Calcul :

# Graphique :

# Réponse :

In [ ]:
# Question 2 :
# Calcul :

# Graphique :

# Réponse :

In [ ]:
# Question 3 :
# Calcul :

# Graphique :

# Réponse :

<details><summary>Solution</summary>

```python
# Question 1 : les légendaires sont-ils vraiment plus forts ?
df.groupby("Legendary")["Total"].mean().plot(kind="bar", title="Total moyen : légendaire ou pas")
plt.show()
# Réponse : oui, environ 640 contre 415 en moyenne.

# Question 2 : quelle génération a les Pokémon les plus rapides ?
df.groupby("Generation")["Speed"].mean().plot(kind="bar", title="Vitesse moyenne par génération")
plt.show()

# Question 3 : les Pokémon rapides sont-ils fragiles ?
plt.scatter(df["Speed"], df["Defense"], alpha=0.5)
plt.xlabel("Vitesse"); plt.ylabel("Défense"); plt.title("Vitesse vs Défense")
plt.show()
# Réponse : pas vraiment de lien clair, le nuage est très dispersé.
```
</details>

## 8. Charger un autre dataset

Le même code marche avec n'importe quel fichier CSV. Sur Kaggle, télécharge un dataset (bouton *Download*, puis dézippe), et charge-le dans Colab :

1. Dans Colab, clique sur l'icône **Fichiers** (dossier, à gauche) → **Importer** → choisis ton `.csv`.
2. Ou exécute la cellule ci-dessous avec `CHARGER_MON_FICHIER = True` : une fenêtre d'envoi s'ouvre.
3. Puis `pd.read_csv("nom_du_fichier.csv")` et c'est parti : `head`, `value_counts`, `groupby`, graphiques.

Datasets gratuits qui plaisent :
- Ventes de jeux vidéo : https://www.kaggle.com/datasets/gregorut/videogamesales
- Films et séries Netflix : https://www.kaggle.com/datasets/shivamb/netflix-shows
- Joueurs FIFA : https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset
- Pokémon (celui de ce notebook) : https://www.kaggle.com/datasets/abcsds/pokemon

In [ ]:
CHARGER_MON_FICHIER = False   # passe à True dans Colab pour envoyer ton propre CSV

if CHARGER_MON_FICHIER:
    try:
        from google.colab import files        # n'existe que dans Colab
        envoye = files.upload()               # ouvre la fenêtre de choix du fichier
        nom_fichier = list(envoye.keys())[0]
        autre = pd.read_csv(nom_fichier)
        print(autre.shape)
        display(autre.head())
    except ImportError:
        print("Pas dans Colab : mets ton CSV à côté du notebook et fais pd.read_csv('nom.csv').")
else:
    print("Mode Pokémon. Passe CHARGER_MON_FICHIER à True pour charger ton propre fichier.")

**Exercice** : si tu as chargé un autre dataset, refais dessus les 4 gestes de base : `head()`, `info()`, un `value_counts()` sur une colonne de catégories, un `groupby` avec une moyenne.
Astuce pour les fichiers qui râlent à l'ouverture : `pd.read_csv("fichier.csv", sep=";")` ou `encoding="latin-1"`.

## À retenir

- Un **DataFrame** = un tableau ; une colonne = une **Series**. `df.head()`, `df.shape`, `df.info()`, `df.describe()` pour faire connaissance.
- **Filtrer** : `df[df["col"] > 10]`, combiner avec `&` et `|`.
- **Trier** : `sort_values("col", ascending=False)`. **Compter** : `value_counts()`.
- **Regrouper** : `df.groupby("catégorie")["nombre"].mean()`.
- Trois graphiques de base : **barres** (comparer), **nuage de points** (lien entre deux nombres), **histogramme** (répartition).
- Une bonne analyse = une **question claire**, un calcul, un graphique, une phrase de réponse.

## Pour montrer aux autres

1. Ta question préférée et son graphique : qu'est-ce qu'on voit en 5 secondes ?
2. Un résultat qui t'a surpris ?
3. Quel geste pandas t'a posé le plus de problèmes (filtre, groupby, graphique) ?

Liens gratuits
- Aide-mémoire pandas (en anglais, 1 page) : https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf
- Galerie matplotlib pour trouver le bon graphique : https://matplotlib.org/stable/gallery/index.html
- Kaggle Datasets : https://www.kaggle.com/datasets